In [ ]:
#linux terminal
#gcloud auth login
#gcloud auth application-default login
#gsutil -m cp -r gs://bucket_name/ save_folder

In [ ]:
#imports
import matplotlib.pyplot as plt
import pandas as pd
#import seaborn as sns
import glob
import json
import numpy as np
import os
import sklearn
from sklearn.metrics import confusion_matrix
from sklearn.metrics import PrecisionRecallDisplay
import warnings
import collections
import math
from google.cloud import bigquery
#import evaluate
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
np.seterr(invalid='ignore', divide='ignore')
#set metrics
COUNT = 'count'
CONF = 'conf'
CORRECT_CONF = 'correct_conf'
INCORRECT_CONF = 'incorrect_conf'
ACC = 'acc'
BIN_ACC = 'bin_acc'
BIN_CONF = 'bin_conf'

In [ ]:
#functions

#save json file
def save_params(hyper_params, save_path):
    json_string = json.dumps(hyper_params)
    with open(save_path, 'w') as outfile:
        outfile.write(json_string)

#load the train params back in
def load_params(fpath):
    # Opening JSON file
    with open(fpath) as json_file:
        data = json.load(json_file)
    return data

#bin initializer
def _bin_initializer(bin_dict, num_bins):
    for i in range(num_bins):
        bin_dict[i][COUNT] = 0
        bin_dict[i][CONF] = 0
        bin_dict[i][CORRECT_CONF] = 0
        bin_dict[i][INCORRECT_CONF] = 0
        bin_dict[i][ACC] = 0
        bin_dict[i][BIN_ACC] = 0
        bin_dict[i][BIN_CONF] = 0

#populate bins
def _populate_bins(confs, preds, labels, num_bins):
    bin_dict = {}
    for i in range(num_bins):
        bin_dict[i] = {}
    _bin_initializer(bin_dict, num_bins)
    num_test_samples = len(confs)

    for i in range(0, num_test_samples):
        confidence = confs[i]
        prediction = preds[i]
        label = labels[i]

        if math.isnan(num_bins * confidence):
            print(confs)
        binn = int(math.ceil(((num_bins * confidence) - 1)))
        # if confidence == 0.0, then binn = -1
        if binn == -1: binn=0

        bin_dict[binn][COUNT] = bin_dict[binn][COUNT] + 1
        bin_dict[binn][CONF] = bin_dict[binn][CONF] + confidence
        bin_dict[binn][CORRECT_CONF] = bin_dict[binn][CORRECT_CONF] + (confidence if (label == prediction) else 0)
        bin_dict[binn][INCORRECT_CONF] = bin_dict[binn][INCORRECT_CONF] + (0 if (label == prediction) else confidence)
        bin_dict[binn][ACC] = bin_dict[binn][ACC] + (1 if (label == prediction) else 0)

    for binn in range(0, num_bins):
        if (bin_dict[binn][COUNT] == 0):
            bin_dict[binn][BIN_ACC] = 0
            bin_dict[binn][BIN_CONF] = 0
        else:
            bin_dict[binn][BIN_ACC] = float(bin_dict[binn][ACC]) / bin_dict[binn][COUNT]
            bin_dict[binn][BIN_CONF] = bin_dict[binn][CONF] / float(bin_dict[binn][COUNT])
    return bin_dict

#expected calibration error
def expected_calibration_error(confs, preds, labels, num_bins):
    bin_dict = _populate_bins(confs, preds, labels, num_bins)
    num_samples = len(labels)
    ece = 0
    bin_stats_dict = collections.defaultdict(dict)
    for i in range(num_bins):
        bin_accuracy = bin_dict[i][BIN_ACC]
        bin_confidence = bin_dict[i][BIN_CONF]
        bin_count = bin_dict[i][COUNT]
        ece += (float(bin_count) / num_samples) * abs(bin_accuracy - bin_confidence)

        # Save the bin stats to be returned
        bin_lower, bin_upper = i*(1/num_bins), (i+1)*(1/num_bins)
        bin_stats_dict[i]['lower_bound'] = bin_lower
        bin_stats_dict[i]['upper_bound'] = bin_upper
        bin_stats_dict[i]['prop_in_bin'] = float(bin_count)/num_samples
        bin_stats_dict[i]['accuracy_in_bin'] = bin_accuracy
        bin_stats_dict[i]['avg_confidence_in_bin'] = bin_confidence
        bin_stats_dict[i]['ece'] = bin_stats_dict[i]['avg_confidence_in_bin'] - bin_stats_dict[i]['accuracy_in_bin']
    #return
    return ece, bin_stats_dict

#max calibration error
def maximum_calibration_error(confs, preds, labels, num_bins):
    bin_dict = _populate_bins(confs, preds, labels, num_bins)
    ce = []
    for i in range(num_bins):
        bin_accuracy = bin_dict[i][BIN_ACC]
        bin_confidence = bin_dict[i][BIN_CONF]
        ce.append(abs(bin_accuracy - bin_confidence))
    return max(ce)
    
#get opt
def get_opt(df, model_config):
    #specify the number of cutoffs
    cutoffs = np.linspace(0.01, 0.99, 99)
    #iterate
    for data in df['DATA'].unique():
        df_sub = df[df['DATA'] == data]
        #init
        metrics = []
        for cutoff in cutoffs:
            #get the predictions above the specified cutoff
            pred_cutoff = np.where(df_sub['Class_1_Prob'] > cutoff, 1, 0)
            #get the confusion matrix based on the cutoff
            tn, fp, fn, tp = confusion_matrix(df_sub[model_config['label']].to_numpy(), pred_cutoff).ravel()
            #calculate precision
            precision = tp/(tp+fp)
            #calculate negative predictive value
            neg_pred_value = tn/(tn+fn)
            #get the recall or sensitivity (binary)
            recall = tp/(tp+fn)
            #get the specificity
            specificity = tn/(tn+fp)
            #get the f1 score
            f1 = (2*precision*recall)/(precision + recall)
            #get the accuracy
            accuracy = (tn+tp)/(tn+fp+fn+tp)
            #collect all metrics
            metrics.append([cutoff, tn, fp, fn, tp, precision, neg_pred_value, recall, specificity, f1, accuracy])

        #visualize the data
        visualize = pd.DataFrame(metrics)
        visualize.columns =['Cutoff', 'True Negative', 'False Positive', 'False Negative','True Positive',
                            'Precision', 'Negative Predictive Value', 'Recall', 'Specificity', 'F1', 'Accuracy']
        #save_file = save_path
        visualize = visualize.sort_values('Accuracy', ascending=False)
        #save file #remove the index
        visualize.to_pickle(model_config['save_folder'] + data + '-OPT.pkl')
        
#get performance
def get_performance(df, model_config, encoding = ['Class 0', 'Class 1']):
    #iterate
    for data in df['DATA'].unique():
        #split data
        df_data = df[df['DATA'] == data]
        #report
        config = sklearn.metrics.classification_report(df_data[model_config['label']].to_numpy().astype(int),
                                                       df_data['Class_Prediction'].to_numpy(), 
                                                       output_dict = True,
                                                       zero_division = 0)
        #save
        model_config[split + '_performance'] = config
        #plot confusion matrix
        plot_confusion_matrix(df_data[model_config['label']].to_numpy().astype(float), 
                              df_data['Class_Prediction'].to_numpy(),
                              split, encoding, model_config)
        #compute
        auc = sklearn.metrics.roc_auc_score(df_data[model_config['label']].to_numpy(),
                                            df_data['Class_1_Prob'].to_numpy(),
                                            multi_class = 'raise')
        #auc
        model_config[split + '_performance']['auc'] = auc
        #compute
        ap = sklearn.metrics.average_precision_score(df_data[model_config['label']].to_numpy(),
                                            df_data['Class_1_Prob'].to_numpy())
        #ap
        model_config[split + '_performance']['ap'] = ap
        #curves
        get_curves(df_data, split, model_config)

    #save the config
    save_params(model_config, model_config['save_folder'] + 'model_config.json')
    #return
    return model_config

#plot confusion matrix
def plot_confusion_matrix(labels, predictions, title, ticks, model_config):
    #cmap
    cm = confusion_matrix(labels, predictions)
    #sub
    ax = plt.subplot()
    sns.set(font_scale=2) # Adjust to fit
    sns.heatmap(cm, annot=True, ax=ax, cmap="binary", fmt="g");
    # Labels, title and ticks
    label_font = {'size':'16'}  # Adjust to fit
    ax.set_xlabel('Predictions', fontdict=label_font);
    ax.set_ylabel('Class Labels', fontdict=label_font);
    title_font = {'size':'16'}  # Adjust to fit
    ax.set_title(title, fontdict=title_font);
    ax.tick_params(axis='both', which='major', labelsize=16)  # Adjust to fit
    if ticks is not None:
        ax.xaxis.set_ticklabels(ticks);
        ax.yaxis.set_ticklabels(ticks);
    #save
    plt.savefig(model_config['save_folder'] + title + '.png', bbox_inches = "tight", dpi = 600)
    plt.close()
    
#nalayze
def analyze(df, model_config):
    ls_metrics = []
    df_col = []
    #tags
    datasets = sorted(df['DATA'].unique())
    #iterate
    for data in datasets:
        #data
        df_data = df[df['DATA'] == data]
        #get metrics
        metrics = model_config[data +'_performance']
        #iterate
        for metric in metrics.keys():
            if type(metrics[metric]) is dict:
                for key in metrics[metric].keys():
                    ls_metrics.append(metrics[metric][key])
                    df_col.append(data + ' ' + metric + ' ' + key)
            else:
                df_col.append(data + ' ' + metric)
                ls_metrics.append(metrics[metric])
    #append
    ls_metrics.append(model_config['save_folder'])
    df_col.append('save_folder')
    #get val_kfold
    #val_kfold = model_config['val_kfold']
    #if list
    #if type(val_kfold) is list:
        #add to ls_metrics
    #    ls_metrics.append(val_kfold[0])
        #add to df cols
    #    df_col.append('val_kfold')
    #return
    return ls_metrics, df_col

#def plot loss
def plot_loss(model_config):
    #if os.path.isdir(model_config['save_folder'] + 'img/') == False:
    #    os.mkdir(model_config['save_folder'] + 'img/')
    f = np.arange(1, len(model_config['log_train_ssl']) + 1)
    plt.plot(f, model_config['log_train_ssl'], label = "Training")
    plt.plot(f, model_config['log_val_ssl'], label = "Validation")
    plt.title('Training & Validation')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    #save
    plt.savefig(model_config['save_folder'] + 'losses.png', 
                bbox_inches = "tight", dpi = 600)
    plt.close()
    
#split 
def get_curves(df, data, model_config):
    y_true = df[model_config['label']].to_list()
    y_pred = df['Class_1_Prob'].to_list()
    PrecisionRecallDisplay.from_predictions(
       y_true, y_pred)
    plt.title(data)
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    #save
    plt.savefig(model_config['save_folder'] + data + '_curve.png', bbox_inches = "tight", dpi = 600)
    plt.close()
    
#cls
def get_cls(row, model_config):
    if row[model_config['label']] == row['Class_Prediction'] == 1:
        label = 'TP'
    elif row[model_config['label']] == row['Class_Prediction'] == 0:
        label = 'TN'
    elif (row[model_config['label']] == 1) and (row['Class_Prediction'] == 0):
        label = 'FN'
    elif (row[model_config['label']] == 0) and (row['Class_Prediction'] == 1):
        label = 'FP'
    return label

#grouped performance
def group_agg(df, 
              col_prob = 'Class_1_Prob',
              col_label = 'Label', 
              col_pred = 'Class_Prediction', 
              col_conf = 'Class_Confidence',
              ls_groupby = ['Group', 'Laterality', 'Date']):
    #return
    return df.groupby(ls_groupby).agg({
        #image laterality here removes the other
        col_prob: lambda x : func_max(x),
        col_pred: lambda x : func_max(x),
        col_conf: lambda x : func_max(x),
        col_label: lambda x : func_max(x),
        'DATA': lambda x : list(x)[0]
        }).reset_index()

#compute_metrics (1)
def compute_metrics(tp, tn, fp, fn):
    try:
        #calculate precision
        precision = tp/(tp+fp)
    except ZeroDivisionError:
        precision = np.nan
    try:
        #calculate negative predictive value
        neg_pred_value = tn/(tn+fn)
    except ZeroDivisionError:
        neg_pred_value = np.nan
    try:
        #get the recall or sensitivity (binary)
        recall = tp/(tp+fn)
    except ZeroDivisionError:
        recall = np.nan
    #get the specificity
    try:
        specificity = tn/(tn+fp)
    except ZeroDivisionError:
        specificity = np.nan
    try:
        #get the f1 score
        f1 = (2*precision*recall)/(precision + recall)
    except:
        f1 = np.nan
    try:
        #get the accuracy
        accuracy = (tn+tp)/(tn+fp+fn+tp)
    except:
        accuracy = np.nan
    #return
    return precision, neg_pred_value, recall, specificity, f1, accuracy

#def macro and weighted 
def mac_wgt(prc_0, npv_0, rec_0, spc_0, f1s_0, prc_1, npv_1, rec_1, spc_1, f1s_1, tp, tn, fp, fn):
    #precision
    mac_prc = 0.5 * (prc_0 + prc_1)
    #negative predictive value
    mac_npv = 0.5 * (npv_0 + npv_1)
    #recall
    mac_rec = 0.5 * (rec_0 + rec_1)
    #specificty
    mac_spc = 0.5 * (spc_0 + spc_1)
    #f1
    mac_f1 = 0.5 * (f1s_0 + f1s_1)
    #weighted average negative
    neg_w = (tn + fp) / (tn+fp+fn+tp)
    #weighted average positive
    pos_w = (tp + fn) / (tn+fp+fn+tp)
    #precision
    wgt_prc = neg_w * prc_0 + pos_w * prc_1
    #negative predictive value
    wgt_npv = neg_w * npv_0 + pos_w * npv_1
    #recall
    wgt_rec = neg_w * rec_0 + pos_w * rec_1
    #spc
    wgt_spc = neg_w * spc_0 + pos_w * spc_1
    #f1
    wgt_f1 = neg_w * f1s_0 + pos_w * f1s_1
    #return 
    return mac_prc, mac_npv, mac_rec, mac_spc, mac_f1, wgt_prc, wgt_npv, wgt_rec, wgt_spc, wgt_f1

#compute_all
def compute_all(df, model_config, groups, num_bins = 10):
    #metrics
    metrics = []
    #data
    for data in df['DATA'].unique():
        #data
        df_data = df[df['DATA'] == data]
        #iterate
        #for mode in list(df_data['Modality'].unique()) + [None]:
        for mode in [None]:
            #subset
            if mode is not None:
                df_mode = df_data[df_data['Modality'] == mode]
            else:
                df_mode = df_data.copy()
            #group
            for group in groups:
                #save_folder
                save_folder = model_config['save_folder']
                #val_kfold + test_kfold
                #if type(model_config['test_kfold']) is list:
                val_fold = model_config['val_fold']
                test_fold = model_config['test_fold']
                #test_kfold
                #elif type(model_config['test_kfold']) is int:
                #    val_fold = None
                #    test_fold = model_config['test_kfold']
                #group aggregation
                df_agg = group_agg(df_mode, 
                                   col_prob = 'Class_1_Prob',
                                   col_label = model_config['label'], 
                                   col_pred = 'Class_Prediction',
                                   col_conf = 'Class_Confidence',
                                   ls_groupby = group)
                #get_cls
                df_agg['CLS'] = df_agg.apply(get_cls, axis = 1, args = (model_config, ))
                #count true positive
                tp = df_agg['CLS'].value_counts().get('TP', 0)
                #true negative
                tn = df_agg['CLS'].value_counts().get('TN', 0)
                #false positive
                fp = df_agg['CLS'].value_counts().get('FP', 0)
                #false negative
                fn = df_agg['CLS'].value_counts().get('FN', 0)
                #binary classification (0)
                prc_0, npv_0, rec_0, spc_0, f1s_0, acc_0 = compute_metrics(tn, tp, fn, fp)
                #binary classification (1)
                prc_1, npv_1, rec_1, spc_1, f1s_1, acc_1 = compute_metrics(tp, tn, fp, fn)
                #average precision score (0)
                aps_0 = sklearn.metrics.average_precision_score(df_agg[model_config['label']], 
                                                              1 - df_agg['Class_1_Prob'])
                #average precision score (1)
                aps_1 = sklearn.metrics.average_precision_score(df_agg[model_config['label']], 
                                                              df_agg['Class_1_Prob'])
                #roc auc score (0)
                auc_0 = sklearn.metrics.roc_auc_score(df_agg[model_config['label']], 
                                                    1 - df_agg['Class_1_Prob'])
                #roc auc score (1)
                auc_1 = sklearn.metrics.roc_auc_score(df_agg[model_config['label']], 
                                                    df_agg['Class_1_Prob'])
                #brier score loss (0)
                bsl_0 = sklearn.metrics.brier_score_loss(df_agg[model_config['label']], 
                                                 1 - df_agg['Class_1_Prob'])
                #brier score loss (1)
                bsl_1 = sklearn.metrics.brier_score_loss(df_agg[model_config['label']], 
                                                         df_agg['Class_1_Prob'])
                #logloss (0)
                log_0 = sklearn.metrics.log_loss(df_agg[model_config['label']], 
                                                         1 - df_agg['Class_1_Prob'], 
                                                 labels = [0, 1])
                #logloss (1)
                log_1 = sklearn.metrics.log_loss(df_agg[model_config['label']], 
                                         df_agg['Class_1_Prob'], 
                                                 labels = [0, 1])
                #ece (0)
                #ece_0 = expected_calibration_error(df_agg['Class_Confidence'], 
                #                                   df_agg['Class_Prediction'], 
                #                                   df_agg[model_config['label']], 
                #                                   num_bins)
                #ece (1)
                #ece_1 = expected_calibration_error(df_agg['Class_Confidence'], 
                #                                   df_agg['Class_Prediction'], 
                #                                   df_agg[model_config['label']], 
                #                                   num_bins)
                #pos count
                pos_count = tp + fn
                #neg count
                neg_count = tn + fp
                #compute overall metrics
                mac_prc, mac_npv, mac_rec, mac_spc, mac_f1, wgt_prc, wgt_npv, wgt_rec, wgt_spc, wgt_f1 = mac_wgt(
                    prc_0, npv_0, rec_0, spc_0, f1s_0, prc_1, npv_1, rec_1, spc_1, f1s_1, tp, tn, fp, fn)
                #select
                if mode is None:
                    mode = 'all'
                else:
                    mode = mode.lower()
                #metrics
                metrics.append([save_folder, val_fold, test_fold, data, mode.lower(), ''.join(group), 
                                tp, tn, fp, fn, 
                                prc_0, npv_0, rec_0, spc_0, f1s_0, acc_0, 
                                prc_1, npv_1, rec_1, spc_1, f1s_1, acc_1, 
                                aps_0, aps_1, auc_0, auc_1, 
                                #ece_0, ece_1, 
                                bsl_0, bsl_1, log_0, log_1,
                                pos_count, neg_count, 
                                mac_prc, mac_npv, mac_rec, mac_spc, mac_f1, 
                                wgt_prc, wgt_npv, wgt_rec, wgt_spc, wgt_f1])         
    #return
    return pd.DataFrame(metrics, columns = [
        'save_folder', 'val_fold', 'test_fold', 'data', 'modality', 'group', 'tp', 'tn', 'fp', 'fn', 
        'prc_0', 'npv_0', 'rec_0', 'spc_0', 'f1_0', 'acc_0',
        'prc_1', 'npv_1', 'rec_1', 'spc_1', 'f1_1', 'acc_1',
        'aps_0', 'aps_1', 'roc_auc_0', 'roc_auc_1', 
        'bsl_0', 'bsl_1', 'log_0', 'log_1',
        'pos_count', 'neg_count', 
        'macro_prc', 'macro_npv', 'macro_rec', 'macro_spc', 'macro_f1',
        'weighted_prc', 'weighted_npv', 'weighted_rec', 'weighted_spc', 'weighted_f1'])

#func_max
def func_max(h):
    return max(list(h))

In [ ]:
#iterate
ls_num = []
#specify the sweep folder
sweep_path = '/'
#sweep_config
sweep_config = load_params(sweep_path + 'sweep_config.json')
#iterate for count
for i in range(sweep_config['count']):
    #specify the model tag
    model_tag = 'trial-' + str(i) +'*'
    #grouping
    groups = [
        ['Sample'], #need an equivalent for document
        #['Group', 'Laterality', 'Date'],
        #['Group', 'Laterality'],
        #['Group']
    ]
    #num_bins 
    num_bins = 10
    #get all the models in the sweep
    sweep_folders = sorted(glob.glob(sweep_path + model_tag + '*'))
    #performance
    performance = []
    #iterate
    for sweep_folder in sweep_folders:
        #load in the params
        model_config = load_params(sweep_folder + '/' + 'model_config.json')
        #load in the results file
        df = pd.read_pickle(model_config['save_folder'] + 'df_results.pkl')
        #workaround
        df['Sample'] = np.arange(0, len(df))
        #compute all
        df_metric = compute_all(df, model_config, groups, num_bins)
        #append
        performance.append(df_metric)
    #concatenate
    df_all = pd.concat(performance).reset_index(drop = True)
    #groupby
    df_num = df_all.groupby(['data', 'group', 'modality'])[[ 
        'prc_1', 'npv_1','rec_1', 'spc_1', 'f1_1', 'acc_1', 'aps_1', 'roc_auc_1', 
        'bsl_1', 'log_1',]].mean().sort_values(
        ['data'], ascending = False).reset_index()
    #save
    #df_num.to_csv(sweep_path + '/df_metric.csv', index = False)
    #round
    df_num.round(2)
    #set
    df_num['model'] = i
    #append
    ls_num.append(df_num)
#concat
df_ = pd.concat(ls_num)

In [ ]:
#trial
df_trial = pd.read_pickle(sweep_config['save_folder'] + 'df_trial.pkl')

In [ ]:
#model number
model_number = df_[df_['data'] == 'VAL'].sort_values(['aps_1'], ascending = False).iloc[0]['model']
display(df_[df_['model'] == model_number].round(2).sort_values(['data']).reset_index(drop = True))
row_val = df_trial[df_trial['number'] == model_number].iloc[0]

In [ ]:
#model number
model_number = df_[df_['data'] == 'TEST'].sort_values(['aps_1'], ascending = False).iloc[0]['model']
display(df_[df_['model'] == model_number].round(2).sort_values(['data']).reset_index(drop = True))
row_test = df_trial[df_trial['number'] == model_number].iloc[0]